# Tarea Hogar 04 — barrido extensivo de LightGBM (`z495`)

Archivo aparte de `z494_TareaHogar_04.ipynb`. No lo pisa. Experimento `HT4950`.

`z494` cruza solo `num_iterations × learning_rate × num_leaves` y **maximiza AUC** de una clase que junta `BAJA+1` y `BAJA+2`. La plata de la materia es ganancia de **BAJA+2** (+975000 / −25000, corte 1/40). Este notebook mide esa ganancia en un holdout estratificado 70/30 de 202107.

## Que dice la doc oficial (y que se pone a prueba aca)

Fuente: [Parameters Tuning](https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html). LightGBM crece **por hojas**, no por profundidad. Por eso overfittea si no frenas `num_leaves` y `min_data_in_leaf`. `max_depth` es un freno extra: `num_leaves` deberia quedar por debajo de `2^max_depth`.

**Vale la pena tunear (y aca se barren):**
- `num_leaves`, `min_data_in_leaf`, `max_depth`, `min_gain_to_split`, `min_sum_hessian_in_leaf` (el `min_child_weight` de XGBoost)
- `learning_rate` **junto** con `num_iterations` (si bajas uno, subis el otro)
- `feature_fraction` y `feature_fraction_bynode`
- `lambda_l1`, `lambda_l2`, `path_smooth`, `extra_trees`
- `bagging_fraction` + `bagging_freq` (si `bagging_freq=0`, el fraction no hace nada)
- desbalance: `scale_pos_weight` **o** `is_unbalance` **o** undersampling de CONTINUA. Los tres a la vez se pisan
- `max_bin` y `min_data_in_bin` (se fijan al armar el `Dataset`)

**Casi no mueven la ganancia, pero la planilla los pide:** `boost_from_average`, `force_col_wise`, `objective=binary`, `first_metric_only`, `drop_rate`/`skip_drop`/`max_drop` (solo importan si `boosting=dart`).

**No hagas una BO de 16 dimensiones.** La propia consigna pregunta si es contraproducente. Aca hay un factor a la vez, unas interacciones chicas, y tres busquedas conjuntas (amplia, arbol chico tipo Denicolay, arbol grande) en el subespacio que la doc dice que importa. `BO iterations` de la planilla = cuantos puntos tuvo esa familia.

**Paralelismo:** PSOCK, no `mclapply`. LightGBM inicializa OpenMP; el fork de `mclapply` se cuelga o se cae. Cada worker usa `num_threads=1` y hay `detectCores()-1` workers (7 en la e2-highmem-8).

**Resume:** graba `trials_th04.tsv` por tanda. Si se corta la VM, re-ejecutar saltea lo ya corrido.

Al final arma `planilla_TH04.tsv` con las columnas de la hoja TareaHogar-04: una fila por familia (la mejor de esa familia).


## 0. Librerias


In [ ]:
# +++ lightgbm puede no estar en la imagen: se instala si falta
if (!require("data.table")) install.packages("data.table")
if (!require("lightgbm")) install.packages("lightgbm")
require("data.table")
require("lightgbm")
require("parallel")

setDTthreads(percent = 100)  # +++ el padre usa todos los cores para fread; los workers van a 1
options(scipen = 999)


## 1. PARAM


In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 271211L   # +++ TU semilla
PARAM$semilla2 <- 552581L             # +++ semilla del undersampling (la "semilla2" de la planilla)
PARAM$estudiante <- "Maceo, Marcos"
PARAM$experimento <- "HT4950"
PARAM$mc_cores <- max(1L, detectCores() - 1L)  # +++ 7 workers en e2-highmem-8
PARAM$mc_cores


## 2. Dataset (solo 202107: ahi hay clase)


In [ ]:
# +++ deteccion de entorno: Colab / VM GCP / local
candidatos_exp <- c("/content/buckets/b1/exp", path.expand("~/buckets/b1/exp"), file.path(getwd(), "exp"))
base_exp <- candidatos_exp[dir.exists(candidatos_exp)][1]
if (is.na(base_exp)) {
  base_exp <- candidatos_exp[3]
  dir.create(base_exp, recursive = TRUE, showWarnings = FALSE)
}
dir.create(file.path(base_exp, PARAM$experimento), showWarnings = FALSE)
setwd(file.path(base_exp, PARAM$experimento))
getwd()

candidatos_ds <- c(
  "/content/datasets/dataset_pequeno.csv",
  path.expand("~/datasets/dataset_pequeno.csv"),
  path.expand("~/buckets/b1/datasets/dataset_pequeno.csv")
)
archivo_dataset <- candidatos_ds[file.exists(candidatos_ds)][1]
stopifnot(!is.na(archivo_dataset))

dataset <- fread(archivo_dataset)
# +++ solo el mes con clase. 202109 queda para el submit final, no para elegir hiperparametros
dataset_mes <- dataset[foto_mes == 202107]
dataset_mes[, c("clase01", "azar", "training") := NULL]
nrow(dataset_mes)
dataset_mes[, .N, clase_ternaria]


## 3. Un job

Entrena en el 70% (con undersampling de CONTINUA), predice el 30%, ganancia normalizada al mes completo. El corte fijo es `1/40`. `mejores_envios` es el cupo que maximiza la ganancia en ese holdout, escalado al mes entero (la columna de la planilla donde Denicolay puso 11000).


In [ ]:
# +++ un job = un modelo LightGBM. Ganancia en holdout 70/30 de 202107 (no AUC).
# +++ PSOCK: esta funcion tiene que ser autocontenida (no usa closures del padre).
eval_job <- function(job) {
  tryCatch({
    data.table::setDTthreads(1)
    t0 <- Sys.time()

    split_strat <- function(dt, p, seed) {
      set.seed(as.integer(seed))
      dwork <- data.table::copy(dt)
      dwork[, `:=`(azar = runif(.N), .rowid = seq_len(.N))]
      data.table::setorderv(dwork, c("clase_ternaria", "azar"))
      dwork[, fold := seq_len(.N) / .N, by = clase_ternaria]
      ids <- dwork[fold <= p, .rowid]
      list(
        train = dt[ids],
        test = dt[setdiff(seq_len(nrow(dt)), ids)]
      )
    }

    sp <- split_strat(dataset_mes, 0.70, job$semilla_split)
    dtrain <- sp$train
    dtest <- sp$test

    set.seed(as.integer(job$semilla2))
    dtrain[, azar_us := runif(.N)]
    dfit <- dtrain[clase_ternaria %in% c("BAJA+1", "BAJA+2") | azar_us <= job$undersampling]

    es <- as.integer(job$early_stopping_rounds)
    dval <- NULL
    if (es > 0L && nrow(dfit) > 1000L) {
      spv <- split_strat(dfit, 0.85, job$semilla_split + 17L)
      dval <- spv$test
      dfit <- spv$train
    }

    lab <- function(cl) {
      if (identical(job$label_mode, "baja1y2")) {
        as.integer(cl %in% c("BAJA+1", "BAJA+2"))
      } else {
        as.integer(cl == "BAJA+2")
      }
    }

    drop_cols <- c(
      "clase_ternaria", "numero_de_cliente", "foto_mes",
      "azar", "azar_us", "fold", ".rowid"
    )
    campos <- setdiff(colnames(dfit), drop_cols)
    num_ok <- vapply(dfit[, campos, with = FALSE], is.numeric, logical(1))
    campos <- campos[num_ok]

    mat_tr <- data.matrix(dfit[, campos, with = FALSE])
    y <- lab(dfit$clase_ternaria)
    ds <- lightgbm::lgb.Dataset(
      data = mat_tr,
      label = y,
      params = list(
        max_bin = as.integer(job$max_bin),
        min_data_in_bin = as.integer(job$min_data_in_bin)
      ),
      free_raw_data = TRUE
    )

    params <- list(
      objective = job$objective,
      metric = "auc",
      boosting = job$boosting,
      num_threads = 1L,
      seed = as.integer(job$seed),
      verbosity = -1,
      learning_rate = job$learning_rate,
      num_leaves = as.integer(job$num_leaves),
      max_depth = as.integer(job$max_depth),
      min_data_in_leaf = as.integer(job$min_data_in_leaf),
      feature_fraction = job$feature_fraction,
      feature_fraction_bynode = job$feature_fraction_bynode,
      bagging_fraction = job$bagging_fraction,
      bagging_freq = as.integer(job$bagging_freq),
      lambda_l1 = job$lambda_l1,
      lambda_l2 = job$lambda_l2,
      min_gain_to_split = job$min_gain_to_split,
      min_sum_hessian_in_leaf = job$min_sum_hessian_in_leaf,
      scale_pos_weight = job$scale_pos_weight,
      is_unbalance = isTRUE(job$is_unbalance),
      boost_from_average = isTRUE(job$boost_from_average),
      extra_trees = isTRUE(job$extra_trees),
      path_smooth = job$path_smooth,
      first_metric_only = TRUE,
      feature_pre_filter = FALSE
    )
    if (isTRUE(job$force_col_wise)) {
      params$force_col_wise <- TRUE
    } else {
      params$force_row_wise <- TRUE
    }
    if (identical(job$boosting, "dart")) {
      params$drop_rate <- job$drop_rate
      params$skip_drop <- job$skip_drop
      params$max_drop <- as.integer(job$max_drop)
    }
    if (job$pos_bagging_fraction < 1 || job$neg_bagging_fraction < 1) {
      params$pos_bagging_fraction <- job$pos_bagging_fraction
      params$neg_bagging_fraction <- job$neg_bagging_fraction
      if (params$bagging_freq == 0L) params$bagging_freq <- 1L
    }

    nrounds <- as.integer(job$num_iterations)
    if (es > 0L && !is.null(dval)) {
      dvalid <- lightgbm::lgb.Dataset(
        data = data.matrix(dval[, campos, with = FALSE]),
        label = lab(dval$clase_ternaria),
        reference = ds
      )
      modelo <- lightgbm::lgb.train(
        params = params,
        data = ds,
        nrounds = nrounds,
        valids = list(valid = dvalid),
        early_stopping_rounds = es,
        verbose = -1
      )
    } else {
      modelo <- lightgbm::lgb.train(
        params = params,
        data = ds,
        nrounds = nrounds,
        verbose = -1
      )
    }

    prob <- predict(modelo, data.matrix(dtest[, campos, with = FALSE]))
    clase <- dtest$clase_ternaria
    gan <- sum(ifelse(
      prob > (1 / 40),
      ifelse(clase == "BAJA+2", 975000, -25000),
      0
    ))
    gan_norm <- gan / 0.30

    ord <- order(prob, decreasing = TRUE)
    cs <- cumsum(ifelse(clase[ord] == "BAJA+2", 975000, -25000))
    k_star <- as.integer(which.max(c(0, cs)) - 1L)
    k_full <- as.integer(round(k_star / 0.30))
    best_iter <- tryCatch(as.integer(modelo$best_iter), error = function(e) nrounds)
    if (length(best_iter) != 1L || is.na(best_iter)) best_iter <- nrounds

    data.table::data.table(
      trial_id = job$trial_id,
      familia = job$familia,
      param_optim = job$param_optim,
      semilla_split = job$semilla_split,
      semilla2 = job$semilla2,
      seed = job$seed,
      undersampling = job$undersampling,
      label_mode = job$label_mode,
      boosting = job$boosting,
      objective = job$objective,
      boost_from_average = job$boost_from_average,
      force_col_wise = job$force_col_wise,
      is_unbalance = job$is_unbalance,
      extra_trees = job$extra_trees,
      num_iterations = nrounds,
      best_iter = best_iter,
      learning_rate = job$learning_rate,
      feature_fraction = job$feature_fraction,
      feature_fraction_bynode = job$feature_fraction_bynode,
      min_data_in_leaf = job$min_data_in_leaf,
      num_leaves = job$num_leaves,
      max_depth = job$max_depth,
      lambda_l1 = job$lambda_l1,
      lambda_l2 = job$lambda_l2,
      min_gain_to_split = job$min_gain_to_split,
      bagging_fraction = job$bagging_fraction,
      bagging_freq = job$bagging_freq,
      pos_bagging_fraction = job$pos_bagging_fraction,
      neg_bagging_fraction = job$neg_bagging_fraction,
      min_sum_hessian_in_leaf = job$min_sum_hessian_in_leaf,
      scale_pos_weight = job$scale_pos_weight,
      drop_rate = job$drop_rate,
      skip_drop = job$skip_drop,
      max_drop = job$max_drop,
      max_bin = job$max_bin,
      min_data_in_bin = job$min_data_in_bin,
      path_smooth = job$path_smooth,
      early_stopping_rounds = es,
      ganancia = gan_norm,
      k_star = k_star,
      mejores_envios = k_full,
      tiempo_seg = round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1),
      error = ""
    )
  }, error = function(e) {
    data.table::data.table(
      trial_id = job$trial_id,
      familia = job$familia,
      param_optim = job$param_optim,
      ganancia = NA_real_,
      error = conditionMessage(e)
    )
  })
}


## 4. Los experimentos

Un factor a la vez sobre un centro (`lr=0.05`, 300 arboles, 31 hojas, `ff=0.8`, undersampling 0.5), mas interacciones mas densas y tres busquedas conjuntas (amplia / arbol chico / arbol grande). Los duplicados exactos del centro se tiran. Dart va capeado a 200 arboles porque es mucho mas lento que `gbdt`.


In [ ]:
# +++ centro del diseno. Cada familia pisa UN eje (o un par acoplado) sobre este base.
# +++ No es el grid de z494 (AUC, 3 params). La metrica la calcula eval_job.
base <- list(
  boosting = "gbdt",
  objective = "binary",
  boost_from_average = TRUE,
  force_col_wise = FALSE,
  is_unbalance = FALSE,
  extra_trees = FALSE,
  num_iterations = 300L,
  learning_rate = 0.05,
  feature_fraction = 0.8,
  feature_fraction_bynode = 1.0,
  min_data_in_leaf = 100L,
  num_leaves = 31L,
  max_depth = -1L,
  lambda_l1 = 0,
  lambda_l2 = 0,
  min_gain_to_split = 0,
  bagging_fraction = 1.0,
  bagging_freq = 0L,
  pos_bagging_fraction = 1.0,
  neg_bagging_fraction = 1.0,
  min_sum_hessian_in_leaf = 0.001,
  scale_pos_weight = 1,
  drop_rate = 0.1,
  skip_drop = 0.5,
  max_drop = 50L,
  max_bin = 31L,
  min_data_in_bin = 3L,
  path_smooth = 0,
  early_stopping_rounds = 0L,
  undersampling = 0.5,
  label_mode = "baja2",
  semilla_split = PARAM$semilla_primigenia,
  semilla2 = PARAM$semilla2,
  seed = PARAM$semilla_primigenia
)

fix_job <- function(job) {
  if (isTRUE(job$bagging_fraction < 1) && job$bagging_freq == 0L) job$bagging_freq <- 1L
  if (isTRUE(job$pos_bagging_fraction < 1 || job$neg_bagging_fraction < 1) && job$bagging_freq == 0L) {
    job$bagging_freq <- 1L
  }
  if (isTRUE(job$is_unbalance)) job$scale_pos_weight <- 1
  if (job$max_depth > 0 && job$num_leaves > (2^job$max_depth - 1)) {
    job$num_leaves <- as.integer(2^job$max_depth - 1)
  }
  if (identical(job$boosting, "dart") && job$num_iterations > 200L) {
    job$num_iterations <- 200L  # +++ dart es mucho mas lento; tope para que el barrido termine
  }
  job
}

one <- function(familia, param_optim, overrides) {
  job <- modifyList(base, overrides)
  job$familia <- familia
  job$param_optim <- param_optim
  fix_job(job)
}

expand_jobs <- function(familia, param_optim, grid) {
  tb <- do.call(CJ, grid)
  lapply(seq_len(nrow(tb)), function(i) {
    ov <- as.list(tb[i])
    one(familia, param_optim, ov)
  })
}

diagonal <- function(familia, param_optim, cols) {
  n <- length(cols[[1]])
  lapply(seq_len(n), function(i) {
    ov <- lapply(cols, function(v) v[[i]])
    one(familia, param_optim, ov)
  })
}

jobs <- list()
push <- function(xs) jobs <<- c(jobs, xs)

# --- referencias (puntos nombrados, no un barrido) ---
push(list(one(
  "00_baseline", "baseline del diseno",
  list()
)))
push(list(one(
  "00_ref_z102", "punto z102 (curso)",
  list(learning_rate = 0.05, num_iterations = 100L, num_leaves = 31L,
       min_data_in_leaf = 100L, feature_fraction = 0.5, max_bin = 31L, undersampling = 1)
)))
push(list(one(
  "00_ref_denicolay", "punto Denicolay planilla",
  list(num_iterations = 1000L, learning_rate = 0.027, feature_fraction = 0.8,
       min_data_in_leaf = 76L, num_leaves = 8L, max_depth = -1L, undersampling = 1)
)))

# --- un factor a la vez: complejidad del arbol ---
push(expand_jobs("num_leaves", "num_leaves", list(
  num_leaves = c(4L, 7L, 11L, 15L, 23L, 31L, 47L, 63L, 95L, 127L, 191L, 255L, 511L)
)))
push(expand_jobs("min_data_in_leaf", "min_data_in_leaf", list(
  min_data_in_leaf = c(5L, 10L, 20L, 40L, 50L, 80L, 100L, 150L, 200L, 300L, 400L, 600L, 800L, 1200L, 1500L, 3000L, 5000L)
)))
push(expand_jobs("max_depth", "max_depth", list(
  max_depth = c(-1L, 2L, 3L, 4L, 5L, 6L, 7L, 8L, 10L, 12L, 14L, 16L, 20L)
)))
push(expand_jobs("min_gain_to_split", "min_gain_to_split", list(
  min_gain_to_split = c(0, 1e-4, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5)
)))
push(expand_jobs("min_sum_hessian_in_leaf", "min_sum_hessian_in_leaf (= min_child_weight de xgboost)", list(
  min_sum_hessian_in_leaf = c(1e-3, 0.01, 0.1, 0.5, 1, 5, 10, 20, 50, 100)
)))
push(expand_jobs("path_smooth", "path_smooth", list(
  path_smooth = c(0, 0.01, 0.1, 1, 5, 10, 20, 50, 100)
)))
push(expand_jobs("extra_trees", "extra_trees", list(
  extra_trees = c(FALSE, TRUE)
)))

# --- learning_rate y num_iterations ---
push(diagonal(
  "lr_x_nrounds",
  "learning_rate + num_iterations (acoplados)",
  list(
    learning_rate = c(0.3, 0.2, 0.1, 0.07, 0.05, 0.03, 0.02, 0.015, 0.01, 0.005),
    num_iterations = c(40L, 80L, 150L, 220L, 300L, 500L, 800L, 1000L, 1500L, 2000L)
  )
))
push(expand_jobs("num_iterations", "num_iterations (lr fijo 0.05)", list(
  num_iterations = c(30L, 50L, 80L, 100L, 150L, 200L, 300L, 400L, 600L, 800L, 1200L, 1600L)
)))
push(expand_jobs("learning_rate", "learning_rate (nrounds fijo 300)", list(
  learning_rate = c(0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.07, 0.08, 0.1, 0.12, 0.15, 0.2, 0.3)
)))

# --- columnas ---
push(expand_jobs("feature_fraction", "feature_fraction", list(
  feature_fraction = c(0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0)
)))
push(expand_jobs("feature_fraction_bynode", "feature_fraction_bynode", list(
  feature_fraction_bynode = c(0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0)
)))

# --- regularizacion ---
push(expand_jobs("lambda_l1", "lambda_l1", list(
  lambda_l1 = c(0, 0.001, 0.01, 0.1, 0.5, 1, 3, 10, 30)
)))
push(expand_jobs("lambda_l2", "lambda_l2", list(
  lambda_l2 = c(0, 0.001, 0.01, 0.1, 0.5, 1, 3, 10, 30)
)))

# --- bagging: grilla, no solo la diagonal ---
push(expand_jobs("bagging", "bagging_fraction + bagging_freq", list(
  bagging_fraction = c(0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
  bagging_freq = c(1L, 3L, 5L)
)))
push(expand_jobs("pos_neg_bagging", "pos_bagging_fraction + neg_bagging_fraction", list(
  pos_bagging_fraction = c(1.0, 0.8, 0.5),
  neg_bagging_fraction = c(1.0, 0.5, 0.2),
  bagging_freq = c(1L)
)))

# --- desbalance ---
push(expand_jobs("scale_pos_weight", "scale_pos_weight", list(
  scale_pos_weight = c(1, 2, 3, 5, 8, 10, 15, 20, 40, 80, 120)
)))
push(expand_jobs("is_unbalance", "is_unbalance", list(
  is_unbalance = c(FALSE, TRUE)
)))
push(expand_jobs("undersampling", "Porcion undersampling CONTINUA", list(
  undersampling = c(0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 0.85, 1.0)
)))

# --- binning ---
push(expand_jobs("max_bin", "max_bin", list(
  max_bin = c(7L, 15L, 31L, 63L, 127L, 255L, 511L)
)))
push(expand_jobs("min_data_in_bin", "min_data_in_bin", list(
  min_data_in_bin = c(1L, 3L, 5L, 10L, 20L, 50L, 100L)
)))

# --- flags de la planilla ---
push(expand_jobs("boost_from_average", "boost_from_average", list(
  boost_from_average = c(TRUE, FALSE)
)))
push(expand_jobs("force_col_wise", "force_col_wise", list(
  force_col_wise = c(FALSE, TRUE)
)))
push(expand_jobs("label_mode", "definicion de la clase positiva", list(
  label_mode = c("baja2", "baja1y2")
)))

# --- dart ---
push(diagonal(
  "dart",
  "boosting=dart + drop_rate/skip_drop/max_drop",
  list(
    boosting = rep("dart", 10),
    drop_rate = c(0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.1, 0.1, 0.1, 0.1),
    skip_drop = c(0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.1, 0.2, 0.8, 0.5),
    max_drop = c(50L, 50L, 50L, 50L, 50L, 50L, 50L, 50L, 50L, 10L),
    num_iterations = rep(200L, 10)
  )
))

# --- early stopping a dos learning rates ---
push(expand_jobs("early_stopping", "early_stopping_rounds", list(
  early_stopping_rounds = c(10L, 20L, 50L, 100L, 200L),
  learning_rate = c(0.03, 0.05, 0.1),
  num_iterations = c(2000L)
)))

# --- ruido: 8 semillas ---
push(diagonal(
  "semilla",
  "semilla (ruido, misma config)",
  list(
    semilla_split = c(271211L, 200177L, 410551L, 552581L, 892237L, 123457L, 654323L, 777011L),
    seed =          c(271211L, 200177L, 410551L, 552581L, 892237L, 123457L, 654323L, 777011L),
    semilla2 =      c(552581L, 200177L, 892237L, 410551L, 271211L, 654323L, 123457L, 200177L)
  )
))

# --- interacciones (cubos chicos, no 16 dims) ---
push(expand_jobs("ix_leaves_x_min_data", "num_leaves x min_data_in_leaf", list(
  num_leaves = c(8L, 15L, 31L, 63L, 127L, 255L),
  min_data_in_leaf = c(20L, 50L, 100L, 200L, 400L, 800L)
)))
push(expand_jobs("ix_ff_x_leaves", "feature_fraction x num_leaves", list(
  feature_fraction = c(0.3, 0.5, 0.7, 0.8, 1.0),
  num_leaves = c(8L, 15L, 31L, 63L, 127L)
)))
push(expand_jobs("ix_depth_x_leaves", "max_depth x num_leaves (cap 2^depth-1)", list(
  max_depth = c(3L, 4L, 6L, 8L, 10L, 12L, 16L, -1L),
  num_leaves = c(8L, 15L, 31L, 63L, 127L)
)))
push(expand_jobs("ix_us_x_spw", "undersampling x scale_pos_weight", list(
  undersampling = c(0.1, 0.2, 0.5, 1.0),
  scale_pos_weight = c(1, 5, 10, 20, 40)
)))
push(expand_jobs("ix_l2_x_leaves", "lambda_l2 x num_leaves", list(
  lambda_l2 = c(0, 0.1, 1, 5, 10),
  num_leaves = c(15L, 31L, 63L, 127L)
)))
push(expand_jobs("ix_l1_x_l2", "lambda_l1 x lambda_l2", list(
  lambda_l1 = c(0, 0.1, 1, 5),
  lambda_l2 = c(0, 0.1, 1, 5)
)))
push(expand_jobs("ix_ff_x_us", "feature_fraction x undersampling", list(
  feature_fraction = c(0.4, 0.6, 0.8, 1.0),
  undersampling = c(0.1, 0.3, 0.5, 1.0)
)))
push(expand_jobs("ix_lr_x_leaves", "learning_rate x num_leaves", list(
  learning_rate = c(0.02, 0.05, 0.1),
  num_leaves = c(8L, 15L, 31L, 63L, 127L),
  num_iterations = c(400L)
)))
push(expand_jobs("ix_bin_x_leaves", "max_bin x num_leaves", list(
  max_bin = c(31L, 63L, 127L, 255L),
  num_leaves = c(15L, 31L, 63L, 127L)
)))
push(expand_jobs("ix_us_x_min_data", "undersampling x min_data_in_leaf", list(
  undersampling = c(0.1, 0.3, 0.5, 1.0),
  min_data_in_leaf = c(20L, 50L, 100L, 300L)
)))
push(expand_jobs("ix_label_x_us", "label_mode x undersampling", list(
  label_mode = c("baja2", "baja1y2"),
  undersampling = c(0.2, 0.5, 1.0)
)))
push(expand_jobs("ix_extra_x_leaves", "extra_trees x num_leaves", list(
  extra_trees = c(FALSE, TRUE),
  num_leaves = c(15L, 31L, 63L, 127L)
)))

# --- conjunta amplia ---
sample_joint <- function(n, familia, param_optim, seed, small = FALSE, deep = FALSE) {
  set.seed(seed)
  out <- vector("list", n)
  for (i in seq_len(n)) {
    if (small) {
      md <- sample(c(-1L, 4L, 6L, 8L), 1)
      nl <- sample(c(4L, 7L, 8L, 11L, 15L, 23L, 31L), 1)
      lr_lo <- 0.01; lr_hi <- 0.06
      nrounds <- sample(c(400L, 600L, 800L, 1000L, 1500L), 1)
    } else if (deep) {
      md <- sample(c(-1L, 8L, 10L, 12L, 16L), 1)
      nl <- sample(c(63L, 95L, 127L, 191L, 255L), 1)
      lr_lo <- 0.02; lr_hi <- 0.1
      nrounds <- sample(c(200L, 300L, 500L, 800L), 1)
    } else {
      md <- sample(c(-1L, 4L, 5L, 6L, 8L, 10L, 12L, 16L), 1)
      nl <- sample(c(7L, 11L, 15L, 23L, 31L, 47L, 63L, 95L, 127L, 191L), 1)
      lr_lo <- 0.01; lr_hi <- 0.15
      nrounds <- sample(c(100L, 200L, 300L, 500L, 800L, 1200L), 1)
    }
    if (md > 0) nl <- min(nl, as.integer(2^md - 1))
    bf <- sample(c(1, 1, 0.9, 0.8, 0.7, 0.5), 1)
    out[[i]] <- one(familia, param_optim, list(
      learning_rate = round(10^runif(1, log10(lr_lo), log10(lr_hi)), 4),
      num_iterations = nrounds,
      num_leaves = nl,
      max_depth = md,
      min_data_in_leaf = sample(c(10L, 20L, 40L, 70L, 100L, 150L, 250L, 400L, 800L, 1500L), 1),
      feature_fraction = sample(c(0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1), 1),
      feature_fraction_bynode = sample(c(0.6, 0.8, 1, 1), 1),
      lambda_l1 = sample(c(0, 0, 0, 0.01, 0.1, 1, 5), 1),
      lambda_l2 = sample(c(0, 0, 0.01, 0.1, 1, 3, 10), 1),
      min_gain_to_split = sample(c(0, 0, 0, 0, 0.01, 0.1), 1),
      min_sum_hessian_in_leaf = sample(c(0.001, 0.1, 1, 10), 1),
      bagging_fraction = bf,
      bagging_freq = if (bf < 1) sample(c(1L, 5L), 1) else 0L,
      scale_pos_weight = sample(c(1, 1, 1, 2, 5, 10, 20), 1),
      undersampling = sample(c(0.1, 0.2, 0.3, 0.5, 0.7, 1), 1),
      max_bin = sample(c(31L, 31L, 63L, 127L, 255L), 1),
      extra_trees = sample(c(FALSE, FALSE, FALSE, TRUE), 1)
    ))
  }
  out
}

push(sample_joint(
  160L, "joint_random",
  "joint: lr,nrounds,leaves,min_data,ff,depth,l1,l2,bagging,us,spw,max_bin",
  PARAM$semilla_primigenia
))
push(sample_joint(
  60L, "joint_arbol_chico",
  "joint zona Denicolay: pocas hojas, lr chico, muchas rondas",
  PARAM$semilla_primigenia + 17L,
  small = TRUE
))
push(sample_joint(
  40L, "joint_arbol_grande",
  "joint zona profunda: muchas hojas",
  PARAM$semilla_primigenia + 91L,
  deep = TRUE
))

# +++ saco duplicados exactos (el centro aparece en varios barridos)
sig_of <- function(job) {
  nms <- sort(setdiff(names(job), c("trial_id", "familia", "param_optim")))
  paste(vapply(nms, function(nm) paste0(nm, "=", paste(job[[nm]], collapse = ",")), character(1)), collapse = "|")
}
seen <- character()
jobs_u <- list()
for (job in jobs) {
  s <- sig_of(job)
  if (s %in% seen) next
  seen <- c(seen, s)
  jobs_u[[length(jobs_u) + 1L]] <- job
}
jobs <- jobs_u
for (i in seq_along(jobs)) jobs[[i]]$trial_id <- i

cat("jobs unicos:", length(jobs), "\n")
print(rbindlist(lapply(jobs, function(j) data.table(familia = j$familia)))[, .N, by = familia][order(-N)])


## 5. Corrida

PSOCK + 1 thread por worker. Progreso por tanda. Si un modelo se cae (parametro no soportado por esa version de lightgbm) queda en `error` y el resto sigue.


In [ ]:
archivo_trials <- "trials_th04.tsv"
tb_trials <- data.table()
done_ids <- integer()
if (file.exists(archivo_trials)) {
  tb_trials <- fread(archivo_trials)
  if ("ganancia" %in% names(tb_trials)) {
    done_ids <- tb_trials[is.finite(ganancia), unique(trial_id)]
  }
}
pending <- Filter(function(j) !(j$trial_id %in% done_ids), jobs)
cat("pendientes:", length(pending), " de ", length(jobs), "\n", sep = "")

if (length(pending) > 0L) {
  cl <- makeCluster(PARAM$mc_cores, type = "PSOCK")
  ok_cluster <- TRUE
  clusterEvalQ(cl, {
    Sys.setenv(OMP_NUM_THREADS = "1", MKL_NUM_THREADS = "1")
    suppressPackageStartupMessages({
      library(data.table)
      library(lightgbm)
    })
    data.table::setDTthreads(1)
    NULL
  })
  assign("eval_job", eval_job, envir = .GlobalEnv)
  assign("dataset_mes", dataset_mes, envir = .GlobalEnv)
  clusterExport(cl, c("eval_job", "dataset_mes"), envir = .GlobalEnv)

  bs <- PARAM$mc_cores
  n_b <- ceiling(length(pending) / bs)
  t0 <- Sys.time()
  for (b in seq_len(n_b)) {
    idx <- ((b - 1L) * bs + 1L):min(b * bs, length(pending))
    res <- parLapply(cl, pending[idx], eval_job)
    tb_new <- rbindlist(res, fill = TRUE)
    tb_trials <- rbindlist(list(tb_trials, tb_new), fill = TRUE)
    fwrite(tb_trials, archivo_trials, sep = "\t")

    n_ok <- tb_new[is.finite(ganancia), .N]
    n_bad <- nrow(tb_new) - n_ok
    mins <- as.numeric(difftime(Sys.time(), t0, units = "mins"))
    cat(sprintf(
      "tanda %d/%d | ok %d | fallos %d | mejor tanda %s | %.1f min | ETA %.1f min\n",
      b, n_b, n_ok, n_bad,
      if (n_ok) format(max(tb_new$ganancia, na.rm = TRUE), big.mark = ",", scientific = FALSE) else "NA",
      mins, if (b < n_b) mins / b * (n_b - b) else 0
    ))
    malos <- tb_new[!is.finite(ganancia) | nzchar(error)]
    if (nrow(malos)) print(malos[, .(trial_id, familia, error)])
    flush.console()
  }
  stopCluster(cl)
  ok_cluster <- FALSE
}

cat("trials con ganancia:", tb_trials[is.finite(ganancia), .N], "\n")


## 6. Que parametro movio la ganancia, y la fila de la planilla

`delta_vs_baseline` > 0: esa familia encontro algo mejor que el centro. Si el rango de una familia es chico, ese parametro no vale la pena tunearlo en la proxima pasada.

`planilla_TH04.tsv` tiene una fila por familia, con el mejor punto de esa familia, en el orden de la hoja.


In [ ]:
tb <- tb_trials[is.finite(ganancia)]
base_g <- tb[familia == "00_baseline", ganancia][1]

por_familia <- tb[, .(
  n = .N,
  ganancia_max = max(ganancia),
  ganancia_mean = mean(ganancia),
  ganancia_min = min(ganancia)
), by = .(familia, param_optim)]
por_familia[, delta_vs_baseline := ganancia_max - base_g]
por_familia[, rango := ganancia_max - ganancia_min]
setorder(por_familia, -delta_vs_baseline)
fwrite(por_familia, "sensibilidad_th04.tsv", sep = "\t")

cat("baseline:", format(base_g, big.mark = ",", scientific = FALSE), "\n\n")
cat("=== familias que MAS mejoran el baseline ===\n")
print(por_familia[delta_vs_baseline > 0][1:min(15, .N)])
cat("\n=== familias que casi no se mueven (rango chico) ===\n")
print(por_familia[order(rango)][1:min(10, .N), .(familia, rango, ganancia_max)])

cat("\n=== top 15 trials ===\n")
setorder(tb, -ganancia)
print(tb[1:min(15, .N), .(
  familia, ganancia, mejores_envios, learning_rate, num_iterations, num_leaves,
  min_data_in_leaf, feature_fraction, max_depth, undersampling, scale_pos_weight, max_bin
)])

# +++ una fila por familia = el trial de mayor ganancia
best <- tb[, .SD[which.max(ganancia)], by = familia]
n_map <- por_familia[, .(familia, n, ganancia_mean)]
best <- n_map[best, on = "familia"]

planilla <- best[, .(
  semilla_primigenia = PARAM$semilla_primigenia,
  `semilla2 (undersampling)` = semilla2,
  `Porción Undesampling` = undersampling,
  `Param a optimizar` = param_optim,
  `# semillas en BO` = fifelse(familia == "semilla", as.integer(n), 1L),
  `BO iterations` = n,
  num_iterations = num_iterations,
  learning_rate = learning_rate,
  feature_fraction = feature_fraction,
  min_data_in_leaf = min_data_in_leaf,
  num_leaves = num_leaves,
  max_depth = max_depth,
  lambda_l1 = lambda_l1,
  lambda_l2 = lambda_l2,
  min_gain_to_split = min_gain_to_split,
  bagging_fraction = bagging_fraction,
  min_sum_hessian_in_leaf = min_sum_hessian_in_leaf,
  bagging_freq = bagging_freq,
  scale_pos_weight = scale_pos_weight,
  boosting = boosting,
  boost_from_average = boost_from_average,
  objective = objective,
  first_metric_only = TRUE,
  drop_rate = drop_rate,
  skip_drop = skip_drop,
  force_col_wise = force_col_wise,
  is_unbalance = is_unbalance,
  max_drop = max_drop,
  max_bin = max_bin,
  n_estimators = num_iterations,
  early_stopping_rounds = early_stopping_rounds,
  min_child_weight = min_sum_hessian_in_leaf,
  feature_fraction_bynode = feature_fraction_bynode,
  bagging_freq_2 = bagging_freq,
  Prueba = ganancia,
  `mejores envios` = mejores_envios,
  `Public Leaderboard` = NA_real_,
  `Promedio de las pruebas` = ganancia_mean,
  Estudiante = PARAM$estudiante
)]
setorder(planilla, -Prueba)
fwrite(planilla, "planilla_TH04.tsv", sep = "\t")
cat("\nplanilla:", nrow(planilla), "filas ->", file.path(getwd(), "planilla_TH04.tsv"), "\n")
planilla[1:min(8, .N), .(`Param a optimizar`, Prueba, `Promedio de las pruebas`, `mejores envios`, num_leaves, learning_rate, feature_fraction)]


## 7. Submit localizado del ganador (opcional)

Entrena sobre **todo** 202107, sin undersampling. `min_data_in_leaf` se escala por `1/undersampling`, como hace `z494`, porque el hiperparametro se eligio sobre una muestra mas chica.

Genera los CSV de varios cupos. Submitea solo 3 para no quemar el limite diario. Despues copia el `publicScore` a la columna Public Leaderboard de esa fila.


In [ ]:
CORRER_KAGGLE <- FALSE  # +++ pasar a TRUE cuando ya viste el ganador local
PARAM$submit_cortes <- c(9000L, 11000L, 13000L)

if (CORRER_KAGGLE) {
  mejor <- tb[which.max(ganancia)]
  print(mejor[, .(familia, ganancia, num_leaves, learning_rate, num_iterations, min_data_in_leaf, feature_fraction, undersampling)])

  dfull <- dataset[foto_mes == 202107]
  dfuture <- dataset[foto_mes == 202109]
  y <- as.integer(dfull$clase_ternaria == "BAJA+2")
  drop_cols <- c("clase_ternaria", "numero_de_cliente", "foto_mes", "clase01", "azar", "training")
  campos <- setdiff(colnames(dfull), drop_cols)
  campos <- campos[vapply(dfull[, campos, with = FALSE], is.numeric, logical(1))]

  # +++ z494: el min_data hallado sobre la muestra undersampleada se reescala al mes completo
  min_data_final <- as.integer(round(mejor$min_data_in_leaf / mejor$undersampling))
  cat("min_data_in_leaf final:", min_data_final, "\n")

  ds <- lgb.Dataset(
    data = data.matrix(dfull[, campos, with = FALSE]),
    label = y,
    params = list(max_bin = as.integer(mejor$max_bin)),
    free_raw_data = TRUE
  )
  params <- list(
    objective = "binary", metric = "auc", boosting = mejor$boosting,
    num_threads = PARAM$mc_cores, seed = PARAM$semilla_primigenia, verbosity = -1,
    learning_rate = mejor$learning_rate,
    num_leaves = as.integer(mejor$num_leaves),
    max_depth = as.integer(mejor$max_depth),
    min_data_in_leaf = min_data_final,
    feature_fraction = mejor$feature_fraction,
    feature_fraction_bynode = mejor$feature_fraction_bynode,
    lambda_l1 = mejor$lambda_l1, lambda_l2 = mejor$lambda_l2,
    min_gain_to_split = mejor$min_gain_to_split,
    min_sum_hessian_in_leaf = mejor$min_sum_hessian_in_leaf,
    bagging_fraction = mejor$bagging_fraction,
    bagging_freq = as.integer(mejor$bagging_freq),
    scale_pos_weight = mejor$scale_pos_weight,
    is_unbalance = isTRUE(mejor$is_unbalance),
    boost_from_average = isTRUE(mejor$boost_from_average),
    extra_trees = isTRUE(mejor$extra_trees),
    path_smooth = mejor$path_smooth,
    feature_pre_filter = FALSE,
    force_row_wise = TRUE
  )
  modelo <- lgb.train(
    params = params, data = ds,
    nrounds = as.integer(if (mejor$best_iter > 0) mejor$best_iter else mejor$num_iterations),
    verbose = -1
  )
  prob <- predict(modelo, data.matrix(dfuture[, campos, with = FALSE]))
  ord <- order(prob, decreasing = TRUE)

  for (envios in seq(8000L, 14000L, by = 1000L)) {
    pred <- integer(length(prob))
    pred[ord[seq_len(envios)]] <- 1L
    archivo <- sprintf("KA495_%05d.csv", envios)
    fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, Predicted = pred), archivo)
    cat("csv", archivo, "\n")
    if (envios %in% PARAM$submit_cortes) {
      linea <- sprintf(
        "kaggle competitions submit -c labo-1-ba-inicial -f %s -m 'TH04 %s envios=%d leaves=%s lr=%s ff=%s'",
        archivo, mejor$familia, envios, mejor$num_leaves, mejor$learning_rate, mejor$feature_fraction
      )
      salida <- system(linea, intern = TRUE)
      cat("  SUBMIT:", salida, "\n")
    }
  }
  flush.console()
}


In [ ]:
# +++ scores de Kaggle, para copiar a Public Leaderboard
if (CORRER_KAGGLE) {
  cat(system("kaggle competitions submissions -c labo-1-ba-inicial", intern = TRUE), sep = "\n")
}
